# 02 - Cleaning

**Energy Transition & Electric Mobility in Portugal**

Turns the raw JSON files from `01_extraction.ipynb` into tidy tables in `data/processed/`:

| Output | One row per | Content |
|---|---|---|
| `ren_daily_production.csv` | day | renewable shares, renewable and non-renewable totals, and energy (MWh) per source |
| `ren_market_prices_daily.csv` | day | average, minimum and maximum price for Portugal and Spain, and a market split flag |
| `eredes_mobility_monthly.csv` | month and municipality | electric mobility energy (MWh) |
| `ine_population_municipalities.csv` | municipality | resident population (INE), when available |

## Setup

In [1]:
import json
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

# Works whether the notebook sits in notebooks/ or in the project root
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

## 1. REN production: from raw JSON to one row per source and day

Each day holds one series per source, with readings in **MW** (power) every 15 minutes. Power becomes **MWh** (energy) by multiplying each reading by 0.25 hours.

For every source and day the table keeps the total energy, the average and peak power, the time of the peak and the number of readings used. A series that is empty for a whole day is skipped rather than filled with zero, because "no data" is not the same as "produced nothing".

In [2]:
with open(DATA_RAW / "ren_production_breakdown_raw.json") as f:
    ren_raw = json.load(f)
print(f"Loaded {len(ren_raw)} days")


def process_ren_day(day_entry):
    """Convert one day of REN data into one row per energy source."""
    time_labels = day_entry["data"]["xAxis"][
        "categories"
    ]  # shared by all series of the day
    rows = []
    for series in day_entry["data"]["series"]:
        readings = [
            (t, v) for t, v in zip(time_labels, series["data"]) if v is not None
        ]
        if not readings:
            continue
        times, values = zip(*readings)
        peak = max(values)
        rows.append(
            {
                "date": day_entry["date"],
                "source": series["name"],
                "energy_mwh": round(sum(values) * 0.25, 2),  # MW x 0.25 h = MWh
                "avg_power_mw": round(sum(values) / len(values), 2),
                "peak_power_mw": peak,
                "peak_time": times[values.index(peak)],
                "n_readings": len(values),
            }
        )
    return rows


df_ren_long = pd.DataFrame([row for entry in ren_raw for row in process_ren_day(entry)])
df_ren_long["date"] = pd.to_datetime(df_ren_long["date"])
df_ren_long.head(10)

Loaded 1827 days


,date,source,energy_mwh,avg_power_mw,peak_power_mw,peak_time,n_readings
0,2021-10-28,Consumption + Storage,151249.80,6302.07,7766.1,13:45,96
1,2021-10-28,Consumption,138728.02,5780.33,7288.5,19:30,96
2,2021-10-28,Pumping,12521.77,521.74,1641.4,13:45,96
3,2021-10-28,Hydro,15925.25,663.55,2796.6,07:30,96
4,2021-10-28,Solar,4670.93,194.62,741.9,12:45,96
5,2021-10-28,Import,37788.43,1574.52,2884.0,03:45,96
6,2021-10-28,Export,1460.65,60.86,577.7,19:15,96
7,2021-10-28,Wind,39690.97,1653.79,3399.4,22:00,96
8,2021-10-28,Natural Gas,45727.72,1905.32,2333.2,07:00,96
9,2021-10-28,Other Thermal,701.35,29.22,30.6,17:30,96


### Quality check: source coverage

Some series only exist in the more recent years (battery flows, for example), so a source can appear on fewer days than the total.

In [3]:
days_total = df_ren_long["date"].nunique()
coverage = (
    df_ren_long.groupby("source")["date"]
    .nunique()
    .sort_values()
    .to_frame("days_with_data")
    .assign(days_missing=lambda d: days_total - d["days_with_data"])
)
print(
    f"Days: {days_total} | source-day rows: {len(df_ren_long)} | sources: {df_ren_long['source'].nunique()}"
)
coverage

Days: 1827 | source-day rows: 25100 | sources: 15


,days_with_data,days_missing
source,,
Battery Injection,1032,795
Consumption of Batteries,1032,795
Wave,1112,715
Coal,1827,0
Consumption,1827,0
Consumption + Storage,1827,0
Export,1827,0
Biomass,1827,0
Hydro,1827,0


### Quality check: readings per day

A normal day has 96 readings. On the last Sunday of March the clocks go forward and the day has 23 hours (92 readings); on the last Sunday of October it has 25 hours (100 readings). These days are complete, and their energy is still correct because every reading is worth 0.25 hours. Anything else would point to a gap in the source.

In [4]:
def last_sunday(year, month):
    last_day = date(year, month + 1, 1) - timedelta(days=1)
    return last_day - timedelta(days=(last_day.weekday() - 6) % 7)


years = df_ren_long["date"].dt.year.unique()
clock_change = {pd.Timestamp(last_sunday(y, 3)): 92 for y in years}
clock_change |= {pd.Timestamp(last_sunday(y, 10)): 100 for y in years}

expected = df_ren_long["date"].map(clock_change).fillna(96)
unexpected = df_ren_long[df_ren_long["n_readings"] != expected]

clock_days = (
    df_ren_long[df_ren_long["date"].isin(clock_change)]
    .groupby("date")["n_readings"]
    .max()
)
print(
    f"Clock-change days in the data: {len(clock_days)} "
    f"({(clock_days == 92).sum()} with 92 readings, {(clock_days == 100).sum()} with 100)"
)
print(f"Source-days with an unexpected number of readings: {len(unexpected)}")
unexpected.head()

Clock-change days in the data: 11 (5 with 92 readings, 6 with 100)
Source-days with an unexpected number of readings: 0


,date,source,energy_mwh,avg_power_mw,peak_power_mw,peak_time,n_readings


## 2. One row per day and the renewable share

Only **production** sources count towards the renewable share. Consumption, imports, exports, pumping and battery flows describe demand or flows, not production.

| Group | Sources |
|---|---|
| Renewable | Hydro, Solar, Wind, Biomass, Wave |
| Non-renewable | Natural Gas, Other Thermal, Coal |

Two versions of the share are kept on purpose:

- `pct_renewable`: renewable share of **domestic production** (imports ignored).
- `pct_renewable_incl_imports`: a conservative version that adds imports to the denominator and counts all imported electricity as non-renewable.

Official figures work with the import balance, so they usually fall somewhere between the two. A source missing on a given day counts as 0 in the sums. Wave stays in the list for completeness, although REN reports it as zero on every day.

The REN series names are turned into snake_case column names with an `_mwh` suffix.

In [5]:
SOURCE_COLUMNS = {
    "Hydro": "hydro_mwh",
    "Wind": "wind_mwh",
    "Solar": "solar_mwh",
    "Biomass": "biomass_mwh",
    "Wave": "wave_mwh",
    "Natural Gas": "natural_gas_mwh",
    "Other Thermal": "other_thermal_mwh",
    "Coal": "coal_mwh",
    "Import": "import_mwh",
    "Export": "export_mwh",
    "Consumption": "consumption_mwh",
    "Consumption + Storage": "consumption_storage_mwh",
    "Pumping": "pumping_mwh",
    "Consumption of Batteries": "battery_charging_mwh",
    "Battery Injection": "battery_discharge_mwh",
}
RENEWABLE = ["hydro_mwh", "solar_mwh", "wind_mwh", "biomass_mwh", "wave_mwh"]
NON_RENEWABLE = ["natural_gas_mwh", "other_thermal_mwh", "coal_mwh"]

df_daily = df_ren_long.pivot(index="date", columns="source", values="energy_mwh")

new_series = set(df_daily.columns) - set(SOURCE_COLUMNS)
assert not new_series, f"REN added new series, map them in SOURCE_COLUMNS: {new_series}"

# Every known source becomes a column, even if absent in this window (all NaN)
df_daily = df_daily.reindex(columns=list(SOURCE_COLUMNS)).rename(columns=SOURCE_COLUMNS)
df_daily.columns.name = None

df_daily["renewable_mwh"] = df_daily[RENEWABLE].sum(axis=1)
df_daily["non_renewable_mwh"] = df_daily[NON_RENEWABLE].sum(axis=1)
df_daily["total_production_mwh"] = (
    df_daily["renewable_mwh"] + df_daily["non_renewable_mwh"]
)

supply_with_imports = df_daily["total_production_mwh"] + df_daily["import_mwh"].fillna(
    0
)
df_daily["pct_renewable"] = (
    df_daily["renewable_mwh"] / df_daily["total_production_mwh"] * 100
).round(1)
df_daily["pct_renewable_incl_imports"] = (
    df_daily["renewable_mwh"] / supply_with_imports * 100
).round(1)
df_daily["import_share_pct"] = (
    df_daily["import_mwh"].fillna(0) / supply_with_imports * 100
).round(1)

SHARE_COLUMNS = ["pct_renewable", "pct_renewable_incl_imports", "import_share_pct"]
TOTAL_COLUMNS = ["renewable_mwh", "non_renewable_mwh", "total_production_mwh"]
df_daily = df_daily[SHARE_COLUMNS + TOTAL_COLUMNS + list(SOURCE_COLUMNS.values())]

core_missing = int(
    df_daily[["hydro_mwh", "wind_mwh", "solar_mwh"]].isna().any(axis=1).sum()
)
print(
    f"Period: {df_daily.index.min().date()} to {df_daily.index.max().date()} ({len(df_daily)} days)"
)
print(f"Days missing hydro, wind or solar: {core_missing}")
# Shares over the whole period are weighted by energy (total MWh over total MWh), not averaged day by day
period_renewable = df_daily["renewable_mwh"].sum()
period_production = df_daily["total_production_mwh"].sum()
period_imports = df_daily["import_mwh"].fillna(0).sum()
print(
    f"Renewable share, domestic production only: {period_renewable / period_production * 100:.1f}%"
)
print(
    f"Renewable share, imports as non-renewable: {period_renewable / (period_production + period_imports) * 100:.1f}%"
)
print(
    f"Import share of total supply:              {period_imports / (period_production + period_imports) * 100:.1f}%"
)

df_daily[SHARE_COLUMNS + TOTAL_COLUMNS].head()

Period: 2021-10-28 to 2026-10-28 (1827 days)
Days missing hydro, wind or solar: 0
Renewable share, domestic production only: 78.8%
Renewable share, imports as non-renewable: 62.3%
Import share of total supply:              21.0%


,pct_renewable,pct_renewable_incl_imports,import_share_pct,renewable_mwh,non_renewable_mwh,total_production_mwh
date,,,,,,
2021-10-28,59.7,45.0,24.7,68830.00,46429.07,115259.07
2021-10-29,74.0,61.6,16.7,94143.28,33048.62,127191.90
2021-10-30,94.2,65.3,30.7,86003.75,5340.23,91343.98
2021-10-31,96.3,79.6,17.3,116392.55,4459.20,120851.75
2021-11-01,88.9,58.4,34.3,74987.23,9382.47,84369.70


In [6]:
df_daily.to_csv(DATA_PROCESSED / "ren_daily_production.csv", index_label="date")
print(f"Saved {len(df_daily)} rows to ren_daily_production.csv")

Saved 1827 rows to ren_daily_production.csv


## 3. E-REDES: electric mobility consumption

The aggregation already happened on the server, so cleaning is light: build a date from `ano` and `mes` (first day of the month), convert kWh to MWh, use English column names and check coverage. The data covers mainland Portugal only.

In [7]:
with open(DATA_RAW / "eredes_mobility_grouped_raw.json") as f:
    eredes_raw = json.load(f)

df_eredes = pd.DataFrame(eredes_raw)
df_eredes["date"] = pd.to_datetime(df_eredes["ano"] + "-" + df_eredes["mes"] + "-01")
df_eredes["energy_mwh"] = (df_eredes["total_energia_kwh"] / 1000).round(3)

df_mobility = (
    df_eredes.rename(columns={"distrito": "district", "concelho": "municipality"})[
        ["date", "district", "municipality", "energy_mwh"]
    ]
    .sort_values(["date", "district", "municipality"])
    .reset_index(drop=True)
)

assert not df_mobility.duplicated(["date", "district", "municipality"]).any(), (
    "Duplicate municipality-month rows"
)
print(f"Rows: {len(df_mobility)}")
print(
    f"Districts: {df_mobility['district'].nunique()} | Municipalities: {df_mobility['municipality'].nunique()}"
)
print(
    f"Months: {df_mobility['date'].min():%Y-%m} to {df_mobility['date'].max():%Y-%m} "
    f"({df_mobility['date'].nunique()} months)"
)

df_mobility.to_csv(DATA_PROCESSED / "eredes_mobility_monthly.csv", index=False)
df_mobility.head()

Rows: 3336
Districts: 18 | Municipalities: 278
Months: 2025-05 to 2026-04 (12 months)


,date,district,municipality,energy_mwh
0,2025-05-01,Aveiro,Albergaria-a-Velha,63.111
1,2025-05-01,Aveiro,Anadia,17.955
2,2025-05-01,Aveiro,Arouca,3.050
3,2025-05-01,Aveiro,Aveiro,134.611
4,2025-05-01,Aveiro,Castelo de Paiva,3.676


## 4. REN market prices

Prices for each country come as 24 hourly values per day up to 29 June 2026 and as 96 quarter-hour values from 30 June 2026. Average, minimum and maximum are calculated the same way for both, so the daily table does not change. They are kept because low prices tend to come with high renewable output.

`market_split` flags days where the Portuguese and Spanish daily averages differ by more than the threshold below. That happens when the interconnection between the two countries is congested and the Iberian market splits into two prices.

In [8]:
MARKET_SPLIT_THRESHOLD_EUR = (
    1.0  # EUR/MWh difference between the PT and ES daily averages
)

with open(DATA_RAW / "ren_market_prices_raw.json") as f:
    prices_raw = json.load(f)
print(f"Loaded {len(prices_raw)} days")


def process_price_day(day_entry):
    """Convert one day of REN price data into a single summary row."""
    row = {"date": day_entry["date"]}
    for series in day_entry["data"]["series"]:
        values = [v for v in series["data"] if v is not None]
        if not values:
            continue
        country = series["name"].lower()  # "pt" or "es"
        row[f"price_avg_{country}"] = round(sum(values) / len(values), 2)
        row[f"price_min_{country}"] = min(values)
        row[f"price_max_{country}"] = max(values)
    return row


df_prices = pd.DataFrame([process_price_day(entry) for entry in prices_raw])
df_prices["date"] = pd.to_datetime(df_prices["date"])
df_prices["market_split"] = (
    df_prices["price_avg_pt"] - df_prices["price_avg_es"]
).abs() > MARKET_SPLIT_THRESHOLD_EUR

print(
    f"Rows: {len(df_prices)} | {df_prices['date'].min().date()} to {df_prices['date'].max().date()}"
)
print(
    f"Days with a market split: {df_prices['market_split'].sum()} ({df_prices['market_split'].mean() * 100:.1f}%)"
)

df_prices.to_csv(DATA_PROCESSED / "ren_market_prices_daily.csv", index=False)
df_prices.head()

Loaded 1827 days
Rows: 1827 | 2021-10-28 to 2026-10-28
Days with a market split: 310 (17.0%)


,date,price_avg_pt,price_min_pt,price_max_pt,price_avg_es,price_min_es,price_max_es,market_split
0,2021-10-28,205.04,169.00,252.33,205.04,169.00,252.33,False
1,2021-10-29,175.83,127.05,216.53,175.83,127.05,216.53,False
2,2021-10-30,124.53,58.79,192.07,124.53,58.79,192.07,False
3,2021-10-31,79.00,52.79,148.00,79.00,52.79,148.00,False
4,2021-11-01,91.20,15.00,183.00,91.20,15.00,183.00,False


### Quality check: price resolution and an external cross-check

The number of values per day shows the switch from hourly to quarter-hour prices (and the clock-change days). The monthly averages are then compared with the Portuguese averages published by OMIE, the Iberian market operator, in its [monthly report for July 2026](https://www.omie.es/sites/default/files/2026-08/informe-mensual_072026_pt.pdf). OMIE uses arithmetic averages too, so the two should match closely.

In [ ]:
values_per_day = pd.Series(
    {entry["date"]: len(entry["data"]["series"][0]["data"]) for entry in prices_raw}
)
switch_day = values_per_day[values_per_day >= 92].index.min()
print("Values per day (number of days):")
print(values_per_day.value_counts().sort_index().to_string())
print(f"First day with quarter-hour prices: {switch_day}")

# Monthly averages for Portugal published by OMIE (EUR/MWh)
OMIE_PT_MONTHLY = {"2026-06": 70.25, "2026-07": 106.98}
monthly_pt = df_prices.groupby(df_prices["date"].dt.to_period("M"))[
    "price_avg_pt"
].mean()
for month, reference in OMIE_PT_MONTHLY.items():
    period = pd.Period(month)
    if period in monthly_pt.index:
        print(
            f"{month}: {monthly_pt[period]:.2f} in this project vs {reference:.2f} published by OMIE"
        )

Values per day (number of days):
23        5
24     1696
25        5
96      120
100       1
First day with quarter-hour prices: 2026-06-30
2026-06: 70.27 in this project vs 70.25 published by OMIE
2026-07: 107.05 in this project vs 106.98 published by OMIE


## 5. INE: resident population by municipality

Only runs if `01_extraction.ipynb` managed to download the population. The total for each place is its largest value across sex and age groups, and names are matched to the E-REDES municipalities. Mainland INE codes start with `1`, which settles names that also exist in the islands.

In [ ]:
import unicodedata


def name_key(name):
    """Upper case, no accents, single spaces: used to match place names across sources."""
    plain = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    return " ".join(plain.upper().replace("-", " ").split())


population_raw_path = DATA_RAW / "ine_population_raw.json"
if population_raw_path.exists():
    with open(population_raw_path) as f:
        ine = json.load(f)

    totals = {}
    for record in ine["records"]:
        try:
            value = float(record["valor"])
        except (KeyError, TypeError, ValueError):
            continue
        place = record.get("geocod")
        if place not in totals or value > totals[place][1]:
            totals[place] = (record.get("geodsg", ""), value)

    df_places = pd.DataFrame(
        [{"geocod": str(c), "name": n, "population": v} for c, (n, v) in totals.items()]
    )
    df_places["key"] = df_places["name"].map(name_key)

    mainland = (
        df_mobility[["district", "municipality"]]
        .drop_duplicates()
        .assign(key=lambda d: d["municipality"].map(name_key))
    )
    df_places = df_places[df_places["key"].isin(mainland["key"])]
    # Prefer mainland municipality codes (start with 1, seven characters) when a name repeats
    df_places["mainland_score"] = df_places["geocod"].str.startswith("1").astype(
        int
    ) + (df_places["geocod"].str.len() == 7)
    df_places = df_places.sort_values(
        "mainland_score", ascending=False
    ).drop_duplicates("key")

    df_population = mainland.merge(
        df_places[["key", "population"]], on="key", how="left"
    )
    df_population["year"] = ine["year"]
    df_population["indicator"] = ine["indicator"]

    missing = df_population.loc[
        df_population["population"].isna(), "municipality"
    ].tolist()
    print(
        f"INE indicator {ine['indicator']}, year {ine['year']}: "
        f"{df_population['population'].notna().sum()} of {len(df_population)} municipalities matched"
    )
    print(f"Mainland population: {df_population['population'].sum():,.0f}")
    if missing:
        print(f"Without population: {missing}")

    df_population.to_csv(
        DATA_PROCESSED / "ine_population_municipalities.csv", index=False
    )
    display(df_population.head())
else:
    print(
        "No ine_population_raw.json in data/raw, so this step was skipped. "
        "The dashboard will show totals only."
    )

INE indicator 0008273, year 2023: 278 of 278 municipalities matched
Mainland population: 10,700,528


,district,municipality,key,population,year,indicator
0,Aveiro,Albergaria-a-Velha,ALBERGARIA A VELHA,26118.0,2023,0008273
1,Aveiro,Anadia,ANADIA,28831.0,2023,0008273
2,Aveiro,Arouca,AROUCA,21360.0,2023,0008273
3,Aveiro,Aveiro,AVEIRO,89592.0,2023,0008273
4,Aveiro,Castelo de Paiva,CASTELO DE PAIVA,15743.0,2023,0008273


## 6. Output summary

In [11]:
for name in [
    "ren_daily_production.csv",
    "ren_market_prices_daily.csv",
    "eredes_mobility_monthly.csv",
    "ine_population_municipalities.csv",
]:
    if not (DATA_PROCESSED / name).exists():
        print(f"{name:34s} not created")
        continue
    rows, cols = pd.read_csv(DATA_PROCESSED / name).shape
    print(f"{name:30s} {rows:>6} rows x {cols:>2} columns")

ren_daily_production.csv         1827 rows x 22 columns
ren_market_prices_daily.csv      1827 rows x  8 columns
eredes_mobility_monthly.csv      3336 rows x  4 columns
ine_population_municipalities.csv    278 rows x  6 columns
